# Turkey Inflation Lab — tek dosyada araştırma

**Karar:** Geçmiş aylık TÜFE, bir sonraki açıklanacak aylık değişimi basit referanslardan daha iyi tahmin edebiliyor mu?

Bu dosya veri snapshot'ını, kaynak HTML'ini, deney protokolünü, hesaplama kodunu ve referans sonuçları içerir. Repo klonlama, Drive bağlama veya ayrıca CSV yükleme gerekmez. Colab'da **Runtime → Run all** ile sırayla çalıştırın; CPU yeterlidir. İlk hücre eksik/sürümü farklı paketleri kurabilir, bunun için internet gerekir. Python 3.12.14 ile yerelde; 25 Ekim 2026 tarihinde Colab CPU/Python 3.13.16 üzerinde paket kurulumu → oturumu yeniden başlat → Run all akışıyla doğrulandı. Kaydedilmiş çıktılar yerel çalıştırmadan gelir.

**Hedef:** sonraki aylık TÜFE değişimi (%); hata birimi yüzde puan. İlk tarihçe 2005–2014, geliştirme 2015–2023 (108 ay). **2024-01–2026-09 nihai dönem skorlanmaz.** Güncel veri sürümüyle pseudo-out-of-sample değerlendirmesidir; tarihsel yayım/vintage bilgisi eksiktir. Önceki ayın verisinin bilindiği varsayılır.

Bu notebook scripts/build_notebook.py ile mevcut Python kaynaklarından üretilir. Kod değişikliğinde notebook'u elle değiştirmek yerine yeniden üretin. Kaynak kod ve notebook alternatif kullanım yollarıdır.


In [1]:
import sys, subprocess, importlib.metadata
if sys.version_info < (3, 12):
    raise RuntimeError('Python 3.12+ seçin; yerel kabul Python 3.12.14 ile yapıldı.')
PINS = ['cloudpickle==3.1.2', 'joblib==1.6.0', 'numpy==2.5.4', 'scikit-learn==1.7.2', 'scipy==1.18.1', 'threadpoolctl==3.7.0']
missing=[]
for pin in PINS:
    name,version=pin.split('==')
    try:
        installed=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed=None
    if installed != version:
        missing.append(pin)
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*PINS])
    # Yeni kurulumu önceden yüklenmiş numpy/sklearn ile karıştırmayın.
    if any(name in sys.modules for name in ['numpy','scipy','sklearn']):
        raise RuntimeError('Paketler güncellendi. Runtime → Restart session, sonra Run all yapın.')
print('Python:',sys.version.split()[0])
print('Paket sürümleri:', {p.split('==')[0]:importlib.metadata.version(p.split('==')[0]) for p in PINS})

NOTEBOOK_CODE_SHA256 = {'prepare_cpi.py': 'd66346071aec726b40309d20c0a26bfbddb4792d1114cf158f164cba12b3d6e4', 'evaluate_baselines.py': '1f6a87dd8bdcef2d0df016d124fb65ab535bc82d772c5982ec45ba50bd420ca9', 'evaluate_ridge.py': '94394727c5d865efc04b42401b71085884ac3e4549298a50e77f3609443c2f64'}

DATA_NOTICE = "# Data attribution and reuse\n\nSource: TÜİK (Turkish Statistical Institute), reproduced by the Central Bank of the Republic of Türkiye (TCMB).\n\n- [Original TCMB inflation table](https://tcmb.gov.tr/wps/wcm/connect/TR/TCMB%2BTR/Main%2BMenu/Istatistikler/Enflasyon%2BVerileri)\n- Retrieved: 2026-10-24T21:27:45Z. Coverage: January 2005–September 2026, 261 monthly observations.\n- [TCMB terms](https://www.tcmb.gov.tr/wps/wcm/connect/TR/TCMB%2BTR/Bottom%2BMenu/Diger/Kullanim%2BSartlari), checked 25 October 2026: publication requires attribution; commercial use requires TCMB's written permission.\n- [TÜİK legal notice](https://tuik.gov.tr/Kurumsal/Yasal_Uyari), checked 25 October 2026: reuse of its data is permitted with source attribution.\n\nThis is a noncommercial research portfolio. No blanket open-data license or commercial permission is granted by this repository. Source agency terms continue to apply to the CPI observations and to their inclusion in derived research outputs. The source HTML is preserved solely as the input used for the reproducible extraction; its presentation and third-party assets are not this project's work.\n\nThe CSV normalizes dates and decimal notation; it is not a new official statistical release. Rates are unadjusted monthly/year-on-year percentage changes. Historical publication times and first-release vintages have not been reconstructed. See docs/VERI.md for methodology-label discrepancies.\n\nThe full CSV includes the declared final holdout, which has not been scored or used to select the current models. It is a procedural holdout, not a cryptographically blinded dataset.\n"


Python: 3.12.14
Paket sürümleri: {'cloudpickle': '3.1.2', 'joblib': '1.6.0', 'numpy': '2.5.4', 'scikit-learn': '1.7.2', 'scipy': '1.18.1', 'threadpoolctl': '3.7.0'}


## 1. Taşınabilir veri ve kaynak kontrolü
Sıkıştırılmış bölüm yalnız sabit araştırma dosyalarını taşır; internetten veri indirmez veya kod çalıştırmaz. Açılan CSV ve HTML dosyaları SHA-256 ile denetlenir. Her baştan çalıştırmada ayrı bir work/colab-runs klasörü oluşur. Nihai dönem CSV'de mevcuttur; körlenmiş değildir.


In [2]:
import base64, gzip, hashlib, json, tempfile
from pathlib import Path
BUNDLE_B64 = '...'
base=Path.cwd()/'work'/'colab-runs'
base.mkdir(parents=True,exist_ok=True)
workspace=Path(tempfile.mkdtemp(prefix='inflation-',dir=base))
assets=json.loads(gzip.decompress(base64.b64decode(BUNDLE_B64)))
for name,encoded in assets.items():
    relative=Path(name)
    if relative.is_absolute() or '..' in relative.parts:
        raise ValueError('Geçersiz dosya yolu')
    path=workspace/relative
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_bytes(base64.b64decode(encoded))
checksums=json.loads((workspace/'checksums.json').read_text())
for name,expected in checksums.items():
    assert hashlib.sha256((workspace/name).read_bytes()).hexdigest()==expected,name
print('Doğrulanan paket dosyası:',len(checksums))


Doğrulanan paket dosyası: 9


## 2. Veri okuma ve doğrulama kodu
Aşağıdaki fonksiyonlar mevcut veri hazırlama kodundan aynen alınır. Eksik/tekrar ay, bozuk oran ve belirsiz tablo kabul edilmez. Negatif aylık oranlar korunur.


In [3]:
import argparse

import csv

import hashlib

import json

import re

import sys

from datetime import datetime, timezone

from decimal import Decimal

from html.parser import HTMLParser

from pathlib import Path

SOURCE = "https://tcmb.gov.tr/wps/wcm/connect/TR/TCMB%2BTR/Main%2BMenu/Istatistikler/Enflasyon%2BVerileri"

HEADERS = ("Ay-Yıl", "TÜFE (Yıllık % Değişim)", "TÜFE (Aylık % Değişim)")

MAX_BYTES = 2 * 1024 * 1024

class TableParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.tables = []
        self.table = None
        self.row = None
        self.cell = None

    def handle_starttag(self, tag, attrs):
        if tag == "table":
            if self.table is not None:
                raise ValueError("Nested tables are not supported.")
            self.table = []
        elif self.table is not None:
            if tag == "tr":
                self.row = []
            elif tag in ("td", "th"):
                self.cell = []

    def handle_data(self, data):
        if self.cell is not None:
            self.cell.append(data)

    def handle_endtag(self, tag):
        if tag in ("td", "th") and self.cell is not None:
            if self.row is None:
                raise ValueError("Cell outside a row.")
            self.row.append(" ".join("".join(self.cell).split()))
            self.cell = None
        elif tag == "tr" and self.row is not None:
            if self.cell is not None:
                raise ValueError("Unclosed table cell.")
            self.table.append(self.row)
            self.row = None
        elif tag == "table" and self.table is not None:
            if self.row is not None:
                raise ValueError("Unclosed table row.")
            self.tables.append(self.table)
            self.table = None

def parse_rate(value):
    if len(value) > 32 or not re.fullmatch(r"[+-]?\d+(?:[.,]\d+)?", value):
        raise ValueError("Invalid percentage value.")
    number = Decimal(value.replace(",", "."))
    if not number.is_finite() or number <= -100:
        raise ValueError("CPI percentage change must be finite and greater than -100.")
    return format(number, "f")

def parse_cpi(html):
    if len(html.encode("utf-8")) > MAX_BYTES:
        raise ValueError("HTML exceeds the input limit.")
    parser = TableParser()
    parser.feed(html)
    parser.close()
    if parser.table is not None:
        raise ValueError("Unclosed table.")
    candidates = []
    for table in parser.tables:
        for index, row in enumerate(table):
            if len(row) == 3 and set(row) == set(HEADERS):
                candidates.append((row, table[index + 1:]))
                break
    if len(candidates) != 1:
        raise ValueError("Expected exactly one CPI table with the required headers.")
    header, body = candidates[0]
    positions = [header.index(name) for name in HEADERS]
    records = {}
    for row in body:
        if not row:
            continue
        if len(row) != 3:
            raise ValueError("Unexpected CPI row width.")
        period, annual, monthly = [row[index] for index in positions]
        if not re.fullmatch(r"\d{2}-\d{4}", period):
            raise ValueError("Expected an MM-YYYY reference period.")
        date = datetime.strptime(period, "%m-%Y")
        key = date.strftime("%Y-%m")
        if key in records:
            raise ValueError("Duplicate observation period: " + key)
        records[key] = {
            "observation_period": key,
            "annual_pct": parse_rate(annual),
            "monthly_pct": parse_rate(monthly),
        }
    if not records:
        raise ValueError("CPI table has no observations.")
    ordered = [records[key] for key in sorted(records)]
    months = [int(row["observation_period"][:4]) * 12
              + int(row["observation_period"][5:]) for row in ordered]
    if any(right - left != 1 for left, right in zip(months, months[1:])):
        raise ValueError("Missing observation month.")
    return ordered


## 3. Referans yöntemler ve ölçüm
Son ay, geçen yıl aynı ay ve son 12 ayın aritmetik ortalaması. MAE birincil ölçüt; RMSE büyük hataları, bias yönlü sapmayı gösterir. Yıllık enflasyon ile aylık oran ortalaması aynı kavram değildir.


In [4]:
import argparse

import csv

import hashlib

import io

import json

import math

import platform

import re

import time

from datetime import datetime, timezone

from pathlib import Path

METHODS = ("naive", "seasonal_naive", "trailing_mean_12")

LIMIT = 2 * 1024 * 1024

def month_number(value):
    if not isinstance(value, str) or not re.fullmatch(r"\d{4}-\d{2}", value):
        raise ValueError("Expected a YYYY-MM period.")
    parsed = datetime.strptime(value, "%Y-%m")
    return parsed.year * 12 + parsed.month

def validate_protocol(protocol):
    if protocol["target"] != "monthly_pct" or protocol["primary_metric"] != "MAE":
        raise ValueError("This runner implements monthly CPI and primary MAE only.")
    if protocol["evaluation_mode"] != "latest-vintage-pseudo-out-of-sample":
        raise ValueError("This snapshot cannot support real-time claims.")
    if protocol["methods"] != list(METHODS):
        raise ValueError("The declared three baselines must be evaluated together.")
    names = ("initial_training_start", "initial_training_end", "development_start",
             "development_end", "final_holdout_start", "final_holdout_end")
    start, train_end, dev_start, dev_end, final_start, final_end = [
        month_number(protocol[key]) for key in names]
    if not (train_end - start >= 11 and dev_start == train_end + 1
            and dev_end >= dev_start and final_start == dev_end + 1
            and final_end >= final_start):
        raise ValueError("Training, development and holdout periods must be consecutive.")
    covered = []
    for segment in protocol["subperiods"]:
        left, right = month_number(segment["start"]), month_number(segment["end"])
        if not dev_start <= left <= right <= dev_end:
            raise ValueError("Subperiod is outside development.")
        covered.extend(range(left, right + 1))
    if sorted(covered) != list(range(dev_start, dev_end + 1)):
        raise ValueError("Subperiods must partition development without gaps or overlap.")

def read_limited(path):
    with path.open("rb") as handle:
        data = handle.read(LIMIT + 1)
    if len(data) > LIMIT:
        raise ValueError("Input exceeds size limit.")
    return data

def load_snapshot(snapshot, protocol):
    manifest = json.loads(read_limited(snapshot / "manifest.json"))
    raw_csv = read_limited(snapshot / "cpi.csv")
    digest = hashlib.sha256(raw_csv).hexdigest()
    if digest != manifest["cpi_sha256"] or digest != protocol["expected_csv_sha256"]:
        raise ValueError("CSV hash does not match the snapshot and declared protocol.")
    raw_source = read_limited(snapshot / "source.html")
    if hashlib.sha256(raw_source).hexdigest() != manifest["source_sha256"]:
        raise ValueError("Source snapshot hash mismatch.")
    if manifest["source_url"] != SOURCE or manifest["vintage_status"] != "latest-vintage-only":
        raise ValueError("Unsupported source or vintage status.")
    reader = csv.DictReader(io.StringIO(raw_csv.decode("utf-8")))
    if reader.fieldnames != ["observation_period", "annual_pct", "monthly_pct"]:
        raise ValueError("Unexpected CSV schema.")
    rows = []
    previous = None
    for row in reader:
        period = row["observation_period"]
        number = month_number(period)
        if previous is not None and number != previous + 1:
            raise ValueError("Snapshot periods must be sorted, unique and contiguous.")
        previous = number
        value = float(parse_rate(row["monthly_pct"]))
        rows.append({"period": period, "monthly_pct": value})
    if (not rows or rows[0]["period"] != protocol["initial_training_start"]
            or rows[-1]["period"] != protocol["final_holdout_end"]):
        raise ValueError("Snapshot coverage does not match the declared experiment.")
    return rows, manifest

def predict(history):
    if len(history) < 12 or any(not math.isfinite(value) for value in history):
        raise ValueError("At least 12 finite historical observations are required.")
    return {
        "naive": history[-1],
        "seasonal_naive": history[-12],
        "trailing_mean_12": math.fsum(history[-12:]) / 12,
    }

def walk_forward(rows, protocol):
    periods = [row["period"] for row in rows]
    left = periods.index(protocol["development_start"])
    right = periods.index(protocol["development_end"])
    if left < 12:
        raise ValueError("Not enough pre-development history.")
    predictions = []
    for index in range(left, right + 1):
        history = [row["monthly_pct"] for row in rows[:index]]
        forecasts = predict(history)
        for method in METHODS:
            predictions.append({
                "target_period": rows[index]["period"],
                "origin_period": rows[index - 1]["period"],
                "method": method,
                "actual": rows[index]["monthly_pct"],
                "prediction": forecasts[method],
                "error": forecasts[method] - rows[index]["monthly_pct"],
            })
    return predictions

def summarize(predictions, subperiods, methods=METHODS):
    segments = [{"name": "all_development", "start": predictions[0]["target_period"],
                 "end": predictions[-1]["target_period"]}] + subperiods
    results = []
    for segment in segments:
        for method in methods:
            errors = [row["error"] for row in predictions if row["method"] == method
                      and segment["start"] <= row["target_period"] <= segment["end"]]
            if not errors:
                raise ValueError("Empty evaluation segment.")
            results.append({
                "segment": segment["name"], "method": method, "n": len(errors),
                "mae": math.fsum(abs(error) for error in errors) / len(errors),
                "rmse": math.sqrt(math.fsum(error * error for error in errors) / len(errors)),
                "bias": math.fsum(errors) / len(errors),
            })
    return results


In [5]:
protocol=json.loads((workspace/'results/protocol.json').read_text())
validate_protocol(protocol)
rows,source_manifest=load_snapshot(workspace/'snapshot',protocol)
parsed=parse_cpi((workspace/'snapshot/source.html').read_bytes().decode('utf-8-sig'))
assert len(parsed)==len(rows)==261
assert [r['observation_period'] for r in parsed]==[r['period'] for r in rows]
assert [float(r['monthly_pct']) for r in parsed]==[r['monthly_pct'] for r in rows]
print('Kaynak:',source_manifest['source_url'])
print('İndirme:',source_manifest['retrieved_at'])
print('Kapsam:',rows[0]['period'],rows[-1]['period'],'Gözlem:',len(rows))
print('Negatif aylık oran:',sum(r['monthly_pct']<0 for r in rows))
print('Tasarım:',protocol['evaluation_mode'],'Final dönem: skorlanmayacak')


Kaynak: https://tcmb.gov.tr/wps/wcm/connect/TR/TCMB%2BTR/Main%2BMenu/Istatistikler/Enflasyon%2BVerileri
İndirme: 2026-10-24T21:27:45+00:00
Kapsam: 2005-01 2026-09 Gözlem: 261
Negatif aylık oran: 27
Tasarım: latest-vintage-pseudo-out-of-sample Final dönem: skorlanmayacak


## 4. Ridge — eğitim fold'una özel ön işleme
Son 12 aylık oran; alpha 0.1/1/10/100/1000; üç genişleyen iç zaman fold'u ve her birinde 12 doğrulama ayı. Ölçekleme yalnız eğitimde öğrenilir. İç fold boyunca parametreler sabittir; her doğrulama ayı bir-adım tahmini için önceki gerçekleşmeleri kullanır. Her dış ayda model yeniden eğitilir. Gap=0 önceki aylık verinin bilindiği varsayımına dayanır. 108 × (5 × 3 + 1) = 1.728 fit, tek thread. Ridge kare kayıp kullanır; seçim MAE ile yapılır. Katsayılar nedensel politika etkisi değildir.


In [6]:
import argparse

import csv

import hashlib

import json

import platform

import time

from datetime import datetime, timezone

from pathlib import Path

import numpy as np

import sklearn

from sklearn.linear_model import Ridge

from sklearn.model_selection import TimeSeriesSplit

from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

from threadpoolctl import threadpool_limits

def validate_model(model):
    expected = {'lags': 12, 'alphas': [0.1, 1.0, 10.0, 100.0, 1000.0],
                'inner_splits': 3, 'validation_months': 12,
                'tie_break': 'larger_alpha', 'solver': 'svd',
                'scaling': 'train_fold_only', 'gap': 0}
    if model != expected:
        raise ValueError('This version implements only the declared small Ridge design.')

def supervised(history, lags=12):
    values = np.asarray(history, dtype=float)
    if len(values) < lags + 38 or not np.isfinite(values).all():
        raise ValueError('Insufficient finite history for three 12-month validation folds.')
    features = np.asarray([values[t-lags:t][::-1] for t in range(lags, len(values))])
    return features, values[lags:], values[-lags:][::-1].reshape(1, -1)

def estimator(alpha):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha, solver='svd'))

def forecast(history, model):
    validate_model(model)
    x, y, next_x = supervised(history, model['lags'])
    splitter = TimeSeriesSplit(n_splits=3, test_size=12, gap=0)
    folds = list(splitter.split(x))
    trials = []
    for alpha in model['alphas']:
        losses = []
        for train, validation in folds:
            fitted = estimator(alpha).fit(x[train], y[train])
            losses.append(float(np.mean(np.abs(fitted.predict(x[validation])-y[validation]))))
        trials.append({'alpha': alpha, 'inner_mae': sum(losses)/len(losses)})
    best = min(trials, key=lambda row: (row['inner_mae'], -row['alpha']))
    fitted = estimator(best['alpha']).fit(x, y)
    return float(fitted.predict(next_x)[0]), best, trials

def ridge_walk_forward(rows, protocol):
    predictions, tuning = [], []
    for index, row in enumerate(rows):
        if not protocol['development_start'] <= row['period'] <= protocol['development_end']:
            continue
        history = [past['monthly_pct'] for past in rows[:index]]
        value, best, trials = forecast(history, protocol['model'])
        predictions.append({'target_period': row['period'],
                            'origin_period': rows[index-1]['period'], 'method': 'ridge',
                            'actual': row['monthly_pct'], 'prediction': value,
                            'error': value-row['monthly_pct']})
        for trial in trials:
            tuning.append({'target_period': row['period'], **trial,
                           'selected': trial['alpha'] == best['alpha'],
                           'training_targets': len(history)-12})
    if not predictions:
        raise ValueError('No development origins.')
    return predictions, tuning

def write_csv(path, rows):
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


In [7]:
import time
started=time.perf_counter()
with threadpool_limits(limits=1):
    ridge_predictions,tuning=ridge_walk_forward(rows,protocol)
predictions=sorted(walk_forward(rows,protocol)+ridge_predictions,
                   key=lambda row:(row['target_period'],row['method']))
metrics=summarize(predictions,protocol['subperiods'],(*METHODS,'ridge'))
assert len(predictions)==432 and len(tuning)==540
assert max(r['target_period'] for r in predictions)=='2023-12'
print('Tahmin:',len(predictions),'Ridge fit:',len(ridge_predictions)*16)
print('Hesaplama süresi (yerel ortamınıza bağlı):',round(time.perf_counter()-started,3),'s')
for r in metrics:
    print(f"{r['segment']:16} {r['method']:18} n={r['n']:3} MAE={r['mae']:.4f} RMSE={r['rmse']:.4f} bias={r['bias']:.4f}")


Tahmin: 432 Ridge fit: 1728
Hesaplama süresi (yerel ortamınıza bağlı): 2.938 s
all_development  naive              n=108 MAE=1.2427 RMSE=1.9463 bias=-0.0312
all_development  seasonal_naive     n=108 MAE=1.5235 RMSE=2.7079 bias=-0.4028
all_development  trailing_mean_12   n=108 MAE=1.1692 RMSE=2.0740 bias=-0.2348
all_development  ridge              n=108 MAE=1.1397 RMSE=2.2001 bias=-0.4880
2015-2019        naive              n= 60 MAE=0.9547 RMSE=1.2686 bias=-0.0197
2015-2019        seasonal_naive     n= 60 MAE=0.9457 RMSE=1.4128 bias=-0.0557
2015-2019        trailing_mean_12   n= 60 MAE=0.7293 RMSE=1.0857 bias=-0.0531
2015-2019        ridge              n= 60 MAE=0.6733 RMSE=1.0701 bias=-0.2781
2020-2023        naive              n= 48 MAE=1.6027 RMSE=2.5517 bias=-0.0456
2020-2023        seasonal_naive     n= 48 MAE=2.2458 RMSE=3.7421 bias=-0.8367
2020-2023        trailing_mean_12   n= 48 MAE=1.7192 RMSE=2.8645 bias=-0.4618
2020-2023        ridge              n= 48 MAE=1.7227 RMSE=3.075

## 5. Yayımlanan deneyle sonuç eşitliği
Aynı snapshot/protokol ve aynı hedef ayları karşılaştırılır. İşletim sistemi/BLAS farkları için 1e-10 sayısal tolerans kullanılır. Sadece eğitim/geliştirme çıktıları kontrol edilir; final holdout açılmaz.


In [8]:
with (workspace/'results/predictions.csv').open(newline='',encoding='utf-8') as handle:
    reference=list(csv.DictReader(handle))
assert len(reference)==len(predictions)
for expected,actual in zip(reference,predictions):
    for key in ['target_period','origin_period','method']:
        assert expected[key]==actual[key],key
    for key in ['actual','prediction','error']:
        assert math.isclose(float(expected[key]),actual[key],rel_tol=1e-10,abs_tol=1e-10),key
print('432 tahmin yayımlanmış referansla eşleşti. Final holdout skorlanmadı.')


432 tahmin yayımlanmış referansla eşleşti. Final holdout skorlanmadı.


## 6. İş yorumu ve karar
Ridge genel geliştirme MAE'sinde küçük bir iyileşme verir; fakat 2020–2023'te son ay yöntemini geçemez ve genel RMSE/bias daha zayıftır. Bu sonuç canlı kullanım veya kurumsal bütçe entegrasyonu için yeterli kabul değildir. Sonraki araştırma kur/üretici fiyatı/beklenti gibi ekonomik girdilerin yayım tarihlerini doğrulamalıdır. Şirket gider sepeti ile manşet TÜFE ayrıdır; müşteri faydası henüz ölçülmedi.

## 7. CSV ve araştırma kaydı dışa aktarımı
Çıktılar yalnız bu çalıştırmanın klasörüne yazılır. Sonuç ZIP'inin yolu aşağıda görünür; Colab'ın soldaki Files panelinden indirilebilir. Drive erişimi veya GitHub yazma izni istenmez.


In [9]:
import platform, shutil
result_dir=Path(tempfile.mkdtemp(prefix='output-',dir=workspace))
write_csv(result_dir/'predictions.csv',predictions)
write_csv(result_dir/'metrics.csv',metrics)
write_csv(result_dir/'tuning.csv',tuning)
(result_dir/'protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
run_manifest={'holdout_scored':False,'evaluation_mode':protocol['evaluation_mode'],
              'source_csv_sha256':source_manifest['cpi_sha256'],
              'source_url':source_manifest['source_url'],'retrieved_at':source_manifest['retrieved_at'],
              'python_version':platform.python_version(),
              'dependencies':{p.split('==')[0]:importlib.metadata.version(p.split('==')[0]) for p in PINS},
              'generated_code_sha256':NOTEBOOK_CODE_SHA256}
(result_dir/'manifest.json').write_text(json.dumps(run_manifest,indent=2),encoding='utf-8')
(result_dir/'DATA-NOTICE.md').write_text(DATA_NOTICE,encoding='utf-8')
archive=shutil.make_archive(str(result_dir),'zip',result_dir)
print('CSV/manifest ZIP:',Path(archive).relative_to(Path.cwd()))


CSV/manifest ZIP: work\colab-runs\inflation-vp6l6cki\output-08anpum0.zip


# Data attribution and reuse

Source: TÜİK (Turkish Statistical Institute), reproduced by the Central Bank of the Republic of Türkiye (TCMB).

- [Original TCMB inflation table](https://tcmb.gov.tr/wps/wcm/connect/TR/TCMB%2BTR/Main%2BMenu/Istatistikler/Enflasyon%2BVerileri)
- Retrieved: 2026-10-24T21:27:45Z. Coverage: January 2005–September 2026, 261 monthly observations.
- [TCMB terms](https://www.tcmb.gov.tr/wps/wcm/connect/TR/TCMB%2BTR/Bottom%2BMenu/Diger/Kullanim%2BSartlari), checked 25 October 2026: publication requires attribution; commercial use requires TCMB's written permission.
- [TÜİK legal notice](https://tuik.gov.tr/Kurumsal/Yasal_Uyari), checked 25 October 2026: reuse of its data is permitted with source attribution.

This is a noncommercial research portfolio. No blanket open-data license or commercial permission is granted by this repository. Source agency terms continue to apply to the CPI observations and to their inclusion in derived research outputs. The source HTML is preserved solely as the input used for the reproducible extraction; its presentation and third-party assets are not this project's work.

The CSV normalizes dates and decimal notation; it is not a new official statistical release. Rates are unadjusted monthly/year-on-year percentage changes. Historical publication times and first-release vintages have not been reconstructed. See docs/VERI.md for methodology-label discrepancies.

The full CSV includes the declared final holdout, which has not been scored or used to select the current models. It is a procedural holdout, not a cryptographically blinded dataset.


## Kod lisansı ve yöntem kaynakları
Orijinal kod/belgeler MIT; yukarıdaki üçüncü taraf veri kapsamı ayrıdır.

MIT License

Copyright (c) 2026 Mustafa Yörük

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


Yöntem dayanağı: [Forecast evaluation (2023)](https://doi.org/10.1007/s10618-022-00894-5), [REFORMS (2024)](https://doi.org/10.1126/sciadv.adk3452); API: [scikit-learn 1.7](https://scikit-learn.org/1.7/). Ayrıntılı kaynak okuma kapsamı repodaki araştırma becerisinin sources.md dosyasındadır.

---
**Author:** [Mustafa YÖRÜK](https://github.com/mustafayoruk)  
*Statistics Student @ Hacettepe University*
